# Cellpose Segmentation

Run cellpose directly on LBM data using `lsp.cellpose()`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import hsv_to_rgb

import mbo_utilities as mbo
import lbm_suite2p_python as lsp

plt.style.use('dark_background')
plt.rcParams['figure.dpi'] = 120

# select paths interactively (or edit manually)
RAW_PATH = mbo.select_folder(title="Select raw data folder")
OUTPUT_DIR = Path(r"D:/output/cellpose")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {RAW_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
from mbo_utilities.graphics import simple_selector as ss
# edit these paths
RAW_PATH = Path(r"\\server\path\to\raw\data")
OUTPUT_DIR = Path(r"D:/output/cellpose")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

path_selection = ss.select_files()

print(f"Input: {RAW_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# check data shape
arr = mbo.imread(RAW_PATH, fix_phase=True, use_fft=True)
print(f"Shape: {arr.shape}")
print(f"  frames: {arr.shape[0]}")
print(f"  planes: {arr.shape[1] if len(arr.shape) == 4 else 1}")

## Run Cellpose

In [ ]:
# cellpose parameters
DIAMETER = 4              # expected cell diameter in pixels
CELLPROB_THRESHOLD = -6.0 # lower = more cells detected
FLOW_THRESHOLD = 0.0      # flow error threshold
MIN_SIZE = 2              # min pixels per cell
MAX_SIZE_UM = 35          # max cell diameter in microns

results = lsp.cellpose(
    input_data=RAW_PATH,
    save_path=OUTPUT_DIR,
    planes=None,  # None = all planes, or [1, 2] for specific planes
    projection="max",
    diameter=DIAMETER,
    cellprob_threshold=CELLPROB_THRESHOLD,
    flow_threshold=FLOW_THRESHOLD,
    min_size=MIN_SIZE,
    max_size_um=MAX_SIZE_UM,
    reader_kwargs={"fix_phase": True, "use_fft": True},
)

print(f"\nTotal cells: {results['n_rois']}")

## Load Results

In [ ]:
# load results for a specific plane (0-indexed)
PLANE = 0
cp = lsp.load_cellpose_results(OUTPUT_DIR, plane_idx=PLANE)
masks = cp["masks"]
proj = cp["projection"]

n_cells = int(masks.max())
print(f"Plane {PLANE}: {n_cells} cells")
print(f"Image: {proj.shape}")

## View Results

In [ ]:
def normalize99(img):
    p1, p99 = np.percentile(img, [1, 99])
    return np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)

def mask_overlay(img, masks, alpha=0.4):
    """overlay colored masks on grayscale image"""
    img_norm = normalize99(img)
    rgb = np.stack([img_norm]*3, axis=-1)
    if masks.max() > 0:
        n = masks.max()
        np.random.seed(42)
        colors = np.zeros((n+1, 3))
        for i in range(1, n+1):
            colors[i] = hsv_to_rgb([np.random.rand(), 0.8, 0.9])
        mask_rgb = colors[masks]
        mask_area = masks > 0
        rgb[mask_area] = (1-alpha)*rgb[mask_area] + alpha*mask_rgb[mask_area]
    return np.clip(rgb, 0, 1)

In [ ]:
# full field of view
overlay = mask_overlay(proj, masks)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(normalize99(proj), cmap='gray')
axes[0].set_title('Max Projection')
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title(f'{n_cells} cells detected')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# zoomed view (100x100 from center)
cy, cx = proj.shape[0] // 2, proj.shape[1] // 2
s = 50  # half-size

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(normalize99(proj[cy-s:cy+s, cx-s:cx+s]), cmap='gray')
axes[0].set_title('Projection (100x100)')
axes[0].axis('off')

axes[1].imshow(overlay[cy-s:cy+s, cx-s:cx+s])
axes[1].set_title('With Masks')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Cell Size Distribution

In [ ]:
# compute cell sizes
_, cell_sizes = np.unique(masks[masks > 0], return_counts=True)
diameters = 2 * np.sqrt(cell_sizes / np.pi)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(cell_sizes, bins=50, color='cyan', edgecolor='white', alpha=0.7)
axes[0].axvline(np.median(cell_sizes), color='red', linestyle='--',
                label=f'median={np.median(cell_sizes):.0f}')
axes[0].set_xlabel('Cell Area (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Cell Area')
axes[0].legend()

axes[1].hist(diameters, bins=50, color='lime', edgecolor='white', alpha=0.7)
axes[1].axvline(np.median(diameters), color='red', linestyle='--',
                label=f'median={np.median(diameters):.1f}')
axes[1].set_xlabel('Diameter (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Cell Diameter')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Cells: {len(cell_sizes)}")
print(f"Area: min={cell_sizes.min()}, max={cell_sizes.max()}, median={np.median(cell_sizes):.0f}")
print(f"Diameter: min={diameters.min():.1f}, max={diameters.max():.1f}, median={np.median(diameters):.1f}")

## Summary

In [ ]:
print(f"Input: {RAW_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"")
print(f"Parameters:")
print(f"  diameter: {DIAMETER}")
print(f"  cellprob_threshold: {CELLPROB_THRESHOLD}")
print(f"  flow_threshold: {FLOW_THRESHOLD}")
print(f"  min_size: {MIN_SIZE}")
print(f"  max_size_um: {MAX_SIZE_UM}")
print(f"")
print(f"Results:")
print(f"  Cells: {n_cells}")
print(f"  Median diameter: {np.median(diameters):.1f} px")